In [ ]:
import re
import pandas as pd
from pathlib import Path

df = pd.read_csv('C:/Users/DEVICE/Documents/dts/nigeria-food-prices/data/raw/wfp_food_prices_nga.csv')

print("Original columns:")
print(df.columns.tolist())

df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.to_period('M').dt.to_timestamp()

print("Date range:", df['date'].min(), df['date'].max())


Original columns:
['date', 'admin1', 'admin2', 'market', 'market_id', 'latitude', 'longitude', 'category', 'commodity', 'commodity_id', 'unit', 'priceflag', 'pricetype', 'currency', 'price', 'usdprice']
Date range: 2002-01-15 00:00:00 2026-11-15 00:00:00


In [4]:
#keep retail prices in ngn from April 2022
df = df[
    (df['month'] >= '2022-04-01') &
    (df['pricetype'] == 'Retail') &
    (df['currency'] == 'NGN')
].copy()

In [ ]:
# convert units to kg
def get_kg(unit):
    unit = str(unit).strip().upper()

    match = re.fullmatch(r'([\d.]+)?\s*KG', unit)

    if match:
        return float(match.group(1) or 1)

    match = re.fullmatch(r'([\d.]+)\s*G', unit)

    if match:
        return float(match.group(1)) / 1000

    return None


df['kg'] = df['unit'].apply(get_kg)

df = df.dropna(subset=['kg']).copy()

df = df[df['kg'] > 0]

# convert prices to naira per kg
df['price_ngn'] = df['price'] / df['kg']


In [6]:
# select markets
STATE = {
    'Monday': 'Borno',
    'Abba Gamaram': 'Borno',
    'Budum': 'Borno',
    'Gujba (Buni Yadi)': 'Yobe',
    'Potiskum': 'Yobe',
    'Bade (Gashua)': 'Yobe'
}

# select commodities
COMMODITIES = [
    'Rice (local)',
    'Millet',
    'Sorghum',
    'Yam',
    'Beans (white)',
    'Maize flour'
]

df = df[
    df['market'].isin(STATE) &
    df['commodity'].isin(COMMODITIES)
].copy()

df['state'] = df['market'].map(STATE)

df['zone'] = 'North East'


In [ ]:
# checking for missing months
if df.empty:
    raise ValueError("No data found. Check market and commodity names.")

all_months = pd.date_range(
    '2022-04-01',
    df['month'].max(),
    freq='MS'
)

print("\nMISSING MONTH CHECK")

for (market, commodity), group in df.groupby(
    ['market', 'commodity']
):

    available = set(group['month'])

    missing = [
        d.strftime('%Y-%m')
        for d in all_months
        if d not in available
    ]

    print(
        market,
        "|",
        commodity,
        "| Missing months:",
        len(missing)
    )

    print(missing)



MISSING MONTH CHECK
Abba Gamaram | Beans (white) | Missing months: 20
['2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-12', '2025-06', '2025-07', '2025-08']
Abba Gamaram | Maize flour | Missing months: 20
['2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-12', '2025-06', '2025-07', '2025-08']
Abba Gamaram | Millet | Missing months: 20
['2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-12', '2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-12', '2025-06', '2025-07', '2025-08']
Abba Gamaram | Rice (local) | Missing months: 20
['2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11', '2022-12', '2023-01', '2023-12', '2024-01', '2024-02', '202

In [ ]:
# calculating monthly average prices
monthly = df.groupby(
    ['month', 'zone', 'state', 'market', 'commodity'],
    as_index=False
)['price_ngn'].mean()

In [9]:
# filling missing gaps of up to two months
filled = []

for (market, commodity), group in monthly.groupby(
    ['market', 'commodity']
):

    group = group.sort_values('month')

    zone = group['zone'].iloc[0]
    state = group['state'].iloc[0]

    group = group.set_index('month').asfreq('MS')

    group['market'] = market
    group['commodity'] = commodity
    group['zone'] = zone
    group['state'] = state

    group['price_ngn'] = group['price_ngn'].interpolate(
        method='linear',
        limit=2,
        limit_area='inside'
    )

    filled.append(group.reset_index())


food = pd.concat(filled, ignore_index=True)

food = food.dropna(subset=['price_ngn'])


In [10]:
# arranging the final columns
food = food[
    [
        'month',
        'zone',
        'state',
        'market',
        'commodity',
        'price_ngn'
    ]
]

In [11]:


#checking the cleaned dataset
print("\nFINAL DATASET")

print("Shape:", food.shape)

print("\nFirst month:", food['month'].min())

print("Last month:", food['month'].max())

print("\nRows per market and commodity:")

print(
    food.groupby(['market', 'commodity']).size().unstack()
)


FINAL DATASET
Shape: (1686, 6)

First month: 2022-04-01 00:00:00
Last month: 2026-11-01 00:00:00

Rows per market and commodity:
commodity          Beans (white)  Maize flour  Millet  Rice (local)  Sorghum  \
market                                                                         
Abba Gamaram                  43           43      43            43       43   
Bade (Gashua)                 48           48      48            48       48   
Budum                         43           43      43            43       43   
Gujba (Buni Yadi)             50           50      50            50       50   
Monday                        45           45      45            45       45   
Potiskum                      49           49      55            55       49   

commodity          Yam  
market                  
Abba Gamaram        43  
Bade (Gashua)       48  
Budum               43  
Gujba (Buni Yadi)   50  
Monday              45  
Potiskum            55  


In [ ]:

food.to_csv(
    'C:/Users/DEVICE/Documents/dts/nigeria-food-prices/data/processed/food_clean.csv',
    index=False
)



SUCCESS!
Cleaned dataset saved as food_clean.csv
